# Deepfake Speech Detection Research Demo

This notebook is a reviewer-facing orchestrator for the experiments described in `jathin_aaky.pdf`: axis production/geometry, axis fusion, mini_goat headroom, AASIST cross-family checks, ASVspoof/ITW/generalization checks, and the existing audit suite. It intentionally keeps experiment logic in `experiments/` and uses the checked-in scripts/artifacts rather than creating new audits.

Data-loading scripts use Hugging Face where the original experiment code does so. When a heavyweight raw-data prerequisite is unavailable in this checkout, the notebook validates the existing committed artifact and records a validation flag instead of silently inventing a replacement result.

## Setup: Environment, Paths, and Notebook-Safe Runner

This cell prepares the repo paths, Hugging Face cache/token environment, and a notebook-safe runner for command-line experiment scripts. It monkey-patches `sys.argv` because the existing scripts are CLI programs; it also patches `torch.load(..., weights_only=False)` for trusted local Lightning checkpoints because newer PyTorch versions changed the default checkpoint-loading behavior.

In [1]:
from pathlib import Path
import contextlib
import importlib
import json
import os
import runpy
import sys
import time

ROOT = Path.cwd().resolve()
PAPER_PDF = ROOT / 'jathin_aaky.pdf'
assert PAPER_PDF.exists() and (ROOT / 'experiments').exists(), f'Run from repo root with jathin_aaky.pdf present, got {ROOT}'
EXPERIMENTS = ROOT / 'experiments'
SCRIPTS = EXPERIMENTS / 'scripts'
RESULTS = EXPERIMENTS / 'results'
AUDITS = EXPERIMENTS / 'axis_audits'
MODEL_DIR = ROOT / 'models'
FRESH_MODEL_DIR = ROOT / 'models_fresh'
GOOD_MODEL_DIR = ROOT / 'models' / 'good_models'
LEGACY_GOOD_MODEL_DIR = ROOT / 'good models'
DRIVE_GOOD_MODELS_FOLDER_URL = 'https://drive.google.com/drive/folders/1nDfQpDWIS8tpA5u2kQHeE1SKDpIZ0t6z'
FALLBACK_GOOD_MODELS_FOLDER_URL = 'https://drive.google.com/drive/folders/1sXBohzkTeIUjdRMywBmIzIaVFAwKreno'
GOOD_MODEL_MANIFEST = {
    'robust_goat.ckpt': {'drive_id': '1amWa3pEnFdf3d8Z5nU2cqTiSbApjmhob', 'family': 'ASVspoof2019', 'needed': True},
    'mini_goat-best-epoch=02-val-eer=0.0933.ckpt': {'drive_id': '1p4QQNKtIT24uMyuM3tYzdVvl-uiGKns0', 'family': 'ASVspoof2019/mini_goat', 'needed': True},
    'mlaad_goat-best-epoch=05-val-eer=0.2795.ckpt': {'drive_id': '19-S-cXCPRDmmW2dWzpolLxt35ub0KCIc', 'family': 'MLAAD regular GOAT', 'needed': True},
    'mlaad_robust_goat.ckpt': {'drive_id': '1LkLTKsnwLqQia0L7eK4vDRv8pN9IE7Bc', 'fallback_drive_id': '1MihkPDsQRvHynU6dLP9UjuyNc-76DJrB', 'family': 'MLAAD robust GOAT', 'needed': True},
    'mlaad_robust_goat_seed42-best-epoch=05-val-eer=0.3030.ckpt': {'drive_id': '1KMIf1o37fzqDSXJuWJcVQjDDpbteiqgv', 'family': 'MLAAD robust GOAT', 'needed': True},
    'mlaad_robust_goat_seed1024-best-epoch=03-val-eer=0.2976.ckpt': {'drive_id': '16a6JPg3pZVOoBtynbZPC_iiYn80Zygaq', 'family': 'MLAAD robust GOAT', 'needed': True},
}
AMBIGUOUS_MODEL_PATTERNS = ['e9', 'last', 'smoothing', 'smooth']
SUBMISSION_CHECKPOINT_DIRS = [p for p in [GOOD_MODEL_DIR, MODEL_DIR, FRESH_MODEL_DIR, LEGACY_GOOD_MODEL_DIR] if p.exists()]

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('HF_HOME', str(ROOT / 'data' / 'huggingface'))
os.environ.setdefault('HF_DATASETS_CACHE', str(ROOT / 'data' / 'huggingface' / 'datasets'))
os.environ.setdefault('WANDB_MODE', 'disabled')
secret = ROOT / 'secret.txt'
if secret.exists():
    token = secret.read_text().strip()
    if token:
        os.environ.setdefault('HF_TOKEN', token)
        os.environ.setdefault('HUGGING_FACE_HUB_TOKEN', token)

validation_flags = []
metric_results = []

def dependency_report():
    mods = ['numpy', 'pandas', 'scipy', 'sklearn', 'torch', 'torchaudio', 'transformers', 'datasets', 'huggingface_hub', 'pytorch_lightning']
    rows = []
    for name in mods:
        try:
            mod = importlib.import_module(name)
            rows.append((name, getattr(mod, '__version__', 'installed')))
        except Exception as exc:
            rows.append((name, f'MISSING: {type(exc).__name__}: {exc}'))
    return rows

print('Repo root:', ROOT)
print('Paper PDF:', PAPER_PDF.name)
print('Model dir:', MODEL_DIR)
print('Drive good_models folder:', DRIVE_GOOD_MODELS_FOLDER_URL)
print('Fallback good_models folder:', FALLBACK_GOOD_MODELS_FOLDER_URL)
print('Checkpoints:', {str(d.relative_to(ROOT)): [p.name for p in sorted(d.glob('*.ckpt'))] for d in SUBMISSION_CHECKPOINT_DIRS})
print('HF token present:', bool(os.environ.get('HF_TOKEN')))
for pkg, ver in dependency_report():
    print(f'{pkg:20s} {ver}')

def flag(message):
    print('VALIDATION FLAG:', message)
    validation_flags.append(message)

def patch_torch_load_for_lightning():
    """Notebook compatibility patch for trusted local Lightning checkpoints."""
    try:
        import torch
    except Exception:
        return
    if getattr(torch.load, '_new_demo_patched', False):
        return
    original = torch.load
    def patched_load(*args, **kwargs):
        kwargs.setdefault('weights_only', False)
        return original(*args, **kwargs)
    patched_load._new_demo_patched = True
    torch.load = patched_load

@contextlib.contextmanager
def notebook_argv(script_path, extra_args=None):
    old_argv = sys.argv[:]
    sys.argv = [str(script_path)] + list(extra_args or [])
    try:
        yield
    finally:
        sys.argv = old_argv

def run_script(rel_path, extra_args=None, required_outputs=()):
    """Run an existing repository script in a notebook-safe context."""
    patch_torch_load_for_lightning()
    script_path = ROOT / rel_path
    assert script_path.exists(), f'Missing script: {script_path}'
    print(f'\n>>> running {rel_path}')
    t0 = time.time()
    with notebook_argv(script_path, extra_args):
        runpy.run_path(str(script_path), run_name='__main__')
    print(f'<<< finished {rel_path} in {time.time() - t0:.1f}s')
    for out in required_outputs:
        out_path = ROOT / out
        assert out_path.exists(), f'Expected output missing after {rel_path}: {out_path}'
    return True

def read_json(rel_path):
    return json.loads((ROOT / rel_path).read_text())

def show_csv(rel_path, n=20):
    import pandas as pd
    df = pd.read_csv(ROOT / rel_path)
    display(df.head(n))
    return df

def require_artifacts(paths, reason):
    missing = [p for p in paths if not (ROOT / p).exists()]
    if missing:
        raise FileNotFoundError(f'{reason}; missing artifacts: {missing}')
    return True


Repo root: /home/sagemaker-user/DeepfakeDetectionRenewed
Paper PDF: jathin_aaky.pdf
Model dir: /home/sagemaker-user/DeepfakeDetectionRenewed/models
Drive good_models folder: https://drive.google.com/drive/folders/1nDfQpDWIS8tpA5u2kQHeE1SKDpIZ0t6z
Fallback good_models folder: https://drive.google.com/drive/folders/1sXBohzkTeIUjdRMywBmIzIaVFAwKreno
Checkpoints: {'models/good_models': ['mini_goat-best-epoch=02-val-eer=0.0933.ckpt', 'mlaad_goat-best-epoch=05-val-eer=0.2795.ckpt', 'mlaad_robust_goat.ckpt', 'mlaad_robust_goat_seed1024-best-epoch=03-val-eer=0.2976.ckpt', 'mlaad_robust_goat_seed42-best-epoch=05-val-eer=0.3030.ckpt', 'robust_goat.ckpt'], 'models': ['mlaad_robust_goat-best-epoch=05-val-eer=0.2858.ckpt', 'mlaad_robust_goat.ckpt']}
HF token present: True


numpy                1.26.4
pandas               2.3.3
scipy                1.16.3
sklearn              1.7.2
torch                2.8.0
torchaudio           MISSING: ModuleNotFoundError: No module named 'torchaudio'
transformers         4.57.6
datasets             5.0.0
huggingface_hub      0.36.0
pytorch_lightning    2.6.5


## Paper Targets and Validation Helpers

This cell loads the paper-facing expected numbers from `final_outputs2/summary_assets/key_numbers.json`. Later experiment cells compare outputs against these targets and report mismatches directly instead of tuning or rewriting results.

In [2]:
EXPECTED = read_json('final_outputs2/summary_assets/key_numbers.json')
print(json.dumps(EXPECTED, indent=2))

def metric_check(name, actual, expected, tol=5e-4):
    actual = float(actual); expected = float(expected)
    ok = abs(actual - expected) <= tol
    status = 'OK' if ok else 'MISMATCH'
    print(f'{status:9s} {name:45s} actual={actual:.6f} expected={expected:.6f} tol={tol}')
    row = {'metric': name, 'actual': actual, 'expected': expected, 'tol': tol, 'ok': ok}
    metric_results.append(row)
    return row


{
  "mlaad_wavlm_gat": {
    "baseline_eer_mean": 0.2721,
    "fused_eer_mean": 0.1627,
    "dEER": -0.1094,
    "dEER_p": 0.0
  },
  "mlaad_aasist_zeroshot": {
    "baseline_eer": 0.3759592534495553,
    "fused_eer": 0.11649000122108151
  },
  "mlaad_aasist_ft": {
    "val_eer": 0.159,
    "test_eer": 0.20045109568582511,
    "test_auc": 0.8853932821731494
  },
  "sd_along_mlaad_gat_loso_r2": 0.273,
  "sd_along_aasist_ft_rho": 0.34949762030671594,
  "sd_along_aasist_ft_p": 0.0057652696447435346,
  "p3_rho_wavlm_gat": 0.5989010989010989,
  "p3_p_wavlm_gat": 0.030554114172937866,
  "p3_rho_aasist_zs": 0.6428571428571429,
  "p3_p_aasist_zs": 0.017792450692375696,
  "cos_mlaad_itw": 0.05,
  "itw_wavlm_eer_before": 0.363,
  "itw_aasist_eer_before": 0.486,
  "itw_aasist_fused_eer": 0.16066666666666665,
  "same_domain_cross_arch_rho": 0.5534108937070333,
  "cross_domain_rho_range": [
    0.3,
    0.36
  ],
  "j3_mlaad_lda250_dEER": -0.1360553815285471,
  "j3_itw_lda250_dEER": -0.171094336425

## Experiment: Axis Production and Geometry Law

**Runs/tests:** I2 geometry battery plus J1 centroid/LDA/logreg audit.

**Paper claim supported:** frozen WavLM layer-12 embeddings contain a natural-vs-synthetic axis; system spread/position along that axis predicts detector hardness, and the result is not just an arbitrary linear classifier or detector-logit relabeling.

In [3]:
wave_cache = ROOT / 'outputs' / 'px_wave_cache' / 'i2_full_test_waves.npz'
axis_artifacts = [
    'experiments/results/i2_geometry_battery/i2_stats.json',
    'experiments/results/i2_geometry_battery/utt_features.csv',
    'experiments/results/j1_lda_audit/j1_stats.json',
]
if wave_cache.exists():
    run_script('experiments/scripts/i2_geometry_battery.py', required_outputs=[axis_artifacts[0], axis_artifacts[1]])
    run_script('experiments/scripts/j1_lda_and_audit.py', required_outputs=[axis_artifacts[2]])
else:
    flag('Axis-production scripts require raw MLAAD wave cache outputs/px_wave_cache/i2_full_test_waves.npz; validating committed artifacts instead.')
    require_artifacts(axis_artifacts, 'Axis-production cached validation cannot run')

i2 = read_json('experiments/results/i2_geometry_battery/i2_stats.json')
j1 = read_json('experiments/results/j1_lda_audit/j1_stats.json')
print('I2:', json.dumps(i2, indent=2)[:2500])
print('J1:', json.dumps(j1, indent=2)[:2500])
metric_check('J1 mean-axis fusion dEER', j1['A3_fusion_dEER']['w_mean'], EXPECTED['mlaad_wavlm_gat']['dEER'], tol=0.003)


VALIDATION FLAG: Axis-production scripts require raw MLAAD wave cache outputs/px_wave_cache/i2_full_test_waves.npz; validating committed artifacts instead.
I2: {
  "n_systems": 61,
  "min_utts": 8,
  "detector_eer_full_split": 0.26597151122925333,
  "rog_L12": {
    "loo_r2": -0.015136872433483495,
    "perm_p": 0.09395302348825588,
    "spearman": -0.30777366472765727,
    "rho_smoothness_resid": -0.20761501850872552,
    "p": 0.10837781180958463,
    "rho_language_resid": -0.30777366472765727
  },
  "battery_without_rog_loo_r2": 0.11491163049056907,
  "full_battery_loo_r2": 0.11758016776768898,
  "unique_rog": 0.002668537277119909,
  "best_competitor": {
    "name": "vel_entropy_L9",
    "alone": 0.08328011194286633,
    "with_rog": 0.07013133309686181,
    "partial_rho_rog_given_comp": -0.19984135378106818,
    "p": 0.12255369353843915,
    "partial_rho_comp_given_rog": 0.33120042305658376,
    "p2": 0.009126462639758661
  }
}
J1: {
  "A1_classifier": {
    "w_mean": {
      "AUC": 

{'metric': 'J1 mean-axis fusion dEER',
 'actual': -0.10940772850661729,
 'expected': -0.1094,
 'tol': 0.003,
 'ok': True}

## Experiment: Axis Fusion

**Runs/tests:** I7 system-disjoint axis score fusion on MLAAD.

**Paper claim supported:** `s_fused = z(logit) + lambda z(axis)` improves an under-reading WavLM-GAT detector without detector retraining, reducing MLAAD EER from about 0.272 to 0.163.

In [4]:
i7_artifacts = [
    'experiments/results/i7_axis_fusion/i7_headline.csv',
    'experiments/results/i7_axis_fusion/i7_stats.json',
]
if wave_cache.exists():
    run_script('experiments/scripts/i7_axis_fusion.py', required_outputs=i7_artifacts)
else:
    flag('I7 axis-fusion rerun requires raw MLAAD wave cache; validating committed artifact instead.')
    require_artifacts(i7_artifacts, 'I7 cached validation cannot run')

i7_headline = show_csv('experiments/results/i7_axis_fusion/i7_headline.csv')
i7_stats = read_json('experiments/results/i7_axis_fusion/i7_stats.json')
metric_check('MLAAD WavLM-GAT fusion dEER', i7_stats['dEER'], EXPECTED['mlaad_wavlm_gat']['dEER'], tol=0.003)


VALIDATION FLAG: I7 axis-fusion rerun requires raw MLAAD wave cache; validating committed artifact instead.


,seed,scorer,EER,AUC
0,main,detector,0.265972,0.805725
1,main,fused,0.157103,0.922484
2,s42,detector,0.285478,0.785711
3,s42,fused,0.167922,0.911639
4,s1024,detector,0.264888,0.800684
5,s1024,fused,0.163054,0.917511
6,-,axis_alone,0.187429,0.872479


OK        MLAAD WavLM-GAT fusion dEER                   actual=-0.109419 expected=-0.109400 tol=0.003


{'metric': 'MLAAD WavLM-GAT fusion dEER',
 'actual': -0.10941946967491051,
 'expected': -0.1094,
 'tol': 0.003,
 'ok': True}

## Experiment: mini_goat Headroom Test

**Runs/tests:** existing Workstream E mini_goat scoring/fusion artifacts.

**Paper claim supported:** axis fusion helps when the detector has headroom: the deliberately weak `mini_goat` improves on ASVspoof2019 LA (EER about 0.144 to 0.114), while a strong GOAT detector is near unchanged.

In [5]:
mini_artifacts = [
    'experiments/results/e_mini_goat_fusion/mini_goat_fusion_results.json',
    'experiments/results/e_mini_goat_fusion/headline_comparison.csv',
]
asvspoof_cache = ROOT / 'data' / 'asvspoof_2019_la'
# The script loads ASVspoof2019 from Hugging Face when its local HF cache is available.
if asvspoof_cache.exists() and all((MODEL_DIR / n).exists() for n in ['mini_goat.ckpt', 'robust_goat_seed7.ckpt']):
    try:
        run_script('experiments/results/e_mini_goat_fusion/score_and_fuse_mini_goat.py', required_outputs=mini_artifacts)
    except Exception as exc:
        flag(f'mini_goat rerun failed ({type(exc).__name__}: {exc}); validating committed artifact instead.')
        require_artifacts(mini_artifacts, 'mini_goat cached validation cannot run')
else:
    flag('mini_goat rerun needs Hugging Face ASVspoof cache plus loadable GOAT checkpoints; validating committed artifact instead.')
    require_artifacts(mini_artifacts, 'mini_goat cached validation cannot run')

mini = read_json('experiments/results/e_mini_goat_fusion/mini_goat_fusion_results.json')
show_csv('experiments/results/e_mini_goat_fusion/headline_comparison.csv')
metric_check('mini_goat detector-alone EER', mini['mini_goat']['detector_alone_eer'], 0.14375, tol=0.002)
metric_check('mini_goat fused EER', mini['mini_goat']['fused_eer'], 0.11375, tol=0.002)
metric_check('mini_goat dEER', mini['mini_goat']['dEER_mean'], -0.0300, tol=0.003)


VALIDATION FLAG: mini_goat rerun needs Hugging Face ASVspoof cache plus loadable GOAT checkpoints; validating committed artifact instead.


,detector,detector_alone_eer,axis_alone_eer,fused_eer,dEER_mean,dEER_ci95_lo,dEER_ci95_hi,dEER_p
0,mini_goat,0.143750,0.1725,0.113750,-0.030000,-0.045573,-0.012502,0.000
1,"robust_goat (regated, this run)",0.071250,0.1725,0.075625,0.004375,-0.007500,0.015634,0.559
2,"robust_goat (Workstream C, 3-seed mean, publis...",0.078125,0.1725,0.077708,-0.000417,-0.007495,0.006248,0.798


OK        mini_goat detector-alone EER                  actual=0.143750 expected=0.143750 tol=0.002
OK        mini_goat fused EER                           actual=0.113750 expected=0.113750 tol=0.002
OK        mini_goat dEER                                actual=-0.030000 expected=-0.030000 tol=0.003


{'metric': 'mini_goat dEER',
 'actual': -0.03,
 'expected': -0.03,
 'tol': 0.003,
 'ok': True}

## Experiment: Adaptive Axis Head

**Runs/tests:** J3 axis-adaptive calibration head.

**Paper claim supported:** a small labeled calibration set can estimate a corpus-internal axis and improve frozen detector scores without retraining the backbone, with the strongest gains on shifted/weak regimes.

In [6]:
j3_artifacts = [
    'experiments/results/j3_axis_adaptive/j3_results.csv',
    'experiments/results/j3_axis_adaptive/j3_summary.csv',
]
if asvspoof_cache.exists() and wave_cache.exists():
    run_script('experiments/scripts/j3_axis_adaptive_head.py', required_outputs=j3_artifacts)
else:
    flag('J3 rerun needs Hugging Face ASVspoof cache and MLAAD wave cache; validating committed artifact instead.')
    require_artifacts(j3_artifacts, 'J3 cached validation cannot run')

j3 = show_csv('experiments/results/j3_axis_adaptive/j3_summary.csv', n=50)
row = j3[(j3['dataset'] == 'mlaad') & (j3['head'] == 'lda') & (j3['n_cal'] == 250)]
if len(row):
    metric_check('J3 MLAAD LDA n=250 dEER', row.iloc[0]['dEER'], EXPECTED['j3_mlaad_lda250_dEER'], tol=0.01)


VALIDATION FLAG: J3 rerun needs Hugging Face ASVspoof cache and MLAAD wave cache; validating committed artifact instead.


,dataset,head,n_cal,EER,AUC,EER_det,dEER
0,asvspoof,centroid,5,0.080023,0.976264,0.078125,0.001898
1,asvspoof,centroid,10,0.079671,0.976550,0.078125,0.001546
2,asvspoof,centroid,25,0.085071,0.971433,0.078125,0.006946
3,asvspoof,centroid,50,0.089615,0.968815,0.078125,0.011490
4,asvspoof,centroid,100,0.093512,0.964889,0.078125,0.015387
5,asvspoof,centroid,250,0.098750,0.962734,0.078125,0.020625
6,asvspoof,lda,25,0.080361,0.976617,0.078125,0.002236
7,asvspoof,lda,50,0.081457,0.976242,0.078125,0.003332
8,asvspoof,lda,100,0.080543,0.975738,0.078125,0.002418
9,asvspoof,lda,250,0.082060,0.975765,0.078125,0.003935


OK        J3 MLAAD LDA n=250 dEER                       actual=-0.136055 expected=-0.136055 tol=0.01


## Experiment: AASIST Cross-Family Checks

**Runs/tests:** J5 zero-shot AASIST cross-family evaluation and J6 MLAAD fine-tuned AASIST baseline.

**Paper claim supported:** the geometry/hardness signal is not limited to WavLM-GAT; AASIST is naturally weak cross-domain and axis fusion repairs MLAAD/ITW while adding little where AASIST is already strong.

In [7]:
aasist_assets = [
    ROOT / 'baselines' / 'aasist' / 'models' / 'AASIST.py',
    ROOT / 'baselines' / 'aasist' / 'models' / 'weights' / 'AASIST.pth',
    ROOT / 'baselines' / 'aasist' / 'config' / 'AASIST.conf',
]
j5j6_artifacts = [
    'experiments/results/j5_aasist/j5_results.json',
    'experiments/results/j6_aasist_mlaad/j6_results.json',
]
if all(p.exists() for p in aasist_assets):
    run_script('experiments/scripts/j5_aasist_crossfamily.py', required_outputs=[j5j6_artifacts[0]])
    run_script('experiments/scripts/j6_train_aasist_mlaad.py', required_outputs=[j5j6_artifacts[1]])
else:
    missing = [str(p.relative_to(ROOT)) for p in aasist_assets if not p.exists()]
    flag('AASIST rerun skipped because official external assets are missing: ' + ', '.join(missing))
    require_artifacts(j5j6_artifacts, 'AASIST cached validation cannot run')

j5 = read_json('experiments/results/j5_aasist/j5_results.json')
j6 = read_json('experiments/results/j6_aasist_mlaad/j6_results.json')
print('J5:', json.dumps(j5, indent=2)[:4000])
print('J6:', json.dumps(j6, indent=2)[:3000])
metric_check('AASIST zero-shot MLAAD baseline EER', j5['eer']['mlaad'], EXPECTED['mlaad_aasist_zeroshot']['baseline_eer'], tol=5e-4)
metric_check('AASIST zero-shot MLAAD fused EER', j5['H4']['mlaad']['EER_fused'], EXPECTED['mlaad_aasist_zeroshot']['fused_eer'], tol=5e-4)
metric_check('AASIST fine-tuned test EER', j6['eer_test'], EXPECTED['mlaad_aasist_ft']['test_eer'], tol=5e-4)
metric_check('AASIST fine-tuned sd_along rho', j6['law']['sd_along']['rho'], EXPECTED['sd_along_aasist_ft_rho'], tol=5e-4)


VALIDATION FLAG: AASIST rerun skipped because official external assets are missing: baselines/aasist/models/AASIST.py, baselines/aasist/models/weights/AASIST.pth, baselines/aasist/config/AASIST.conf
J5: {
  "eer": {
    "asvspoof21": 0.07278846153846152,
    "mlaad": 0.3759592534495553,
    "itw": 0.486
  },
  "H1": {
    "P1": {
      "rho": 0.15384615384615383,
      "p": 0.6157988686488814,
      "top3_hits": 1
    },
    "P2": {
      "rho": 0.06593406593406594,
      "p": 0.8305423857560938,
      "top3_hits": 1
    },
    "P3": {
      "rho": 0.6428571428571429,
      "p": 0.017792450692375696,
      "top3_hits": 1
    },
    "P4": {
      "rho": -0.010989010989010988,
      "p": 0.971577682777073,
      "top3_hits": 1
    }
  },
  "H1_family21": {
    "rho": 0.21428571428571427,
    "p": 0.4820544547023683
  },
  "H2": {
    "s_along": {
      "rho": -0.14056054997355893,
      "p": 0.2799299791195811
    },
    "sd_along": {
      "rho": 0.23976731887890004,
      "p": 0.062722

{'metric': 'AASIST fine-tuned sd_along rho',
 'actual': 0.34949762030671594,
 'expected': 0.34949762030671594,
 'tol': 0.0005,
 'ok': True}

## Experiment: ASVspoof21 Prospective/Rotation Check

**Runs/tests:** J4 ASVspoof2021 LA prospective predictions.

**Paper claim supported:** corpus-internal axes can carry position/hardness signal within a target domain, but cross-corpus transferred axes are unreliable because the axis rotates across domains; the paper reports the primary transfer as inconclusive and the stronger P3 result as exploratory.

In [8]:
j4_artifacts = [
    'experiments/results/j4_asvspoof21/j4_results.json',
    'experiments/results/j4_asvspoof21/j4_preregistered_predictions.json',
]
keys = Path('/tmp/keys/LA/CM/trial_metadata.txt')
if keys.exists():
    run_script('experiments/scripts/j4_asvspoof21_prospective.py', required_outputs=j4_artifacts)
else:
    flag('J4 rerun skipped because /tmp/keys/LA/CM/trial_metadata.txt is missing; validating committed artifact instead.')
    require_artifacts(j4_artifacts, 'J4 cached validation cannot run')

j4 = read_json('experiments/results/j4_asvspoof21/j4_results.json')
print(json.dumps(j4, indent=2))
metric_check('ASVspoof21 WavLM P3 rho', j4['P3']['rho'], EXPECTED['p3_rho_wavlm_gat'], tol=5e-4)
metric_check('ASVspoof21 WavLM P3 p', j4['P3']['p'], EXPECTED['p3_p_wavlm_gat'], tol=5e-4)


VALIDATION FLAG: J4 rerun skipped because /tmp/keys/LA/CM/trial_metadata.txt is missing; validating committed artifact instead.
{
  "P1": {
    "rho": 0.25274725274725274,
    "p": 0.40477407127094656,
    "top3_hits": 2
  },
  "P2": {
    "rho": 0.19230769230769232,
    "p": 0.5290679872349875,
    "top3_hits": 2
  },
  "P3": {
    "rho": 0.5989010989010989,
    "p": 0.030554114172937866,
    "top3_hits": 2
  },
  "P4": {
    "rho": -0.3736263736263736,
    "p": 0.20855405979520908,
    "top3_hits": 1
  },
  "overall_eer_meanlogit": 0.31961538461538463
}
OK        ASVspoof21 WavLM P3 rho                       actual=0.598901 expected=0.598901 tol=0.0005
OK        ASVspoof21 WavLM P3 p                         actual=0.030554 expected=0.030554 tol=0.0005


{'metric': 'ASVspoof21 WavLM P3 p',
 'actual': 0.030554114172937866,
 'expected': 0.030554114172937866,
 'tol': 0.0005,
 'ok': True}

## Experiment: Existing Audit Suite

**Runs/tests:** the existing red-team audit scripts in `experiments/axis_audits`; this notebook does not create new audits.

**Paper claim supported:** every quantitative claim is independently recomputed from cached artifacts, including multiplicity control, axis rotation, fusion claims, AASIST checks, reliability, and claim/artifact consistency.

In [9]:
audit_scripts = [
    'audit2_sdalong_claim.py',
    'audit3_asvspoof_prospective.py',
    'audit4_axis_rotation.py',
    'audit5_fusion_claims.py',
    'audit6_itw_speaker.py',
    'audit7_agreement.py',
    'audit8_i1_causal.py',
    'audit9_hardness_reliability.py',
    'audit1_multiplicity.py',
    'audit10_consistency.py',
]
if wave_cache.exists():
    for script in audit_scripts:
        run_script(f'experiments/axis_audits/{script}')
else:
    flag('Audit rerun requires raw MLAAD wave cache; validating committed audit outputs instead.')
    require_artifacts([
        'experiments/axis_audits/audits_outputs/audit1_multiplicity/audit1_results.json',
        'experiments/axis_audits/audits_outputs/audit2_sdalong_claim/audit2_results.json',
        'experiments/axis_audits/audits_outputs/audit3_asvspoof_prospective/audit3_results.json',
        'experiments/axis_audits/audits_outputs/audit4_axis_rotation/audit4_results.json',
        'experiments/axis_audits/audits_outputs/audit5_fusion_claims/audit5_results.json',
        'experiments/axis_audits/audits_outputs/audit6_itw_speaker/audit6_results.json',
        'experiments/axis_audits/audits_outputs/audit7_agreement/audit7_results.json',
        'experiments/axis_audits/audits_outputs/audit8_i1_causal/audit8_results.json',
        'experiments/axis_audits/audits_outputs/audit9_hardness_reliability/audit9_results.json',
        'experiments/axis_audits/audits_outputs/audit10_consistency/audit10_results.json',
    ], 'Audit cached validation cannot run')

verdict = ROOT / 'experiments/axis_audits/audits_outputs/COMPREHENSIVE_VERDICT.md'
if verdict.exists():
    print(verdict.read_text()[:5000])
else:
    flag('Comprehensive audit verdict markdown was not found.')


VALIDATION FLAG: Audit rerun requires raw MLAAD wave cache; validating committed audit outputs instead.
# Red-Team Audit of the Geometric-Axis Hardness Claims (final_outputs2)

**Scope.** Independent recomputation and adversarial stress-testing of every quantitative claim in `final_outputs2` (experiments I1–I7, J1–J6): the sd_along/s_along/s_orth/vel_entropy hardness predictors on MLAAD, the ASVspoof 2021 prospective predictions, the In-The-Wild transfer and per-speaker results, the AASIST replications, the axis-rotation law, the fusion/adaptation results, and the causal intervention battery. All audits run from the cached raw artifacts (embeddings, logits, scores) — none rely on the original analysis code paths. Ten audits; code in `experiments/axis_audits/`, outputs in `experiments/axis_audits/audits_outputs/audit{1..10}_*/`.

---

## 1. What is proven beyond reasonable scrutiny

**P1. sd_along → MLAAD hardness (the core law).** Independently recomputed: LOSO R² = 0.277 (claimed 0.27

## Checkpoint Demonstration: Drive Good Models Load

**Runs/tests:** one representative checkpoint from the local `models/good_models` mirror of the provided Google Drive folder.

**Paper/submission claim supported:** submitted checkpoints should come from the fresh Drive `models/good_models` folder and load as nonempty PyTorch/Lightning state dicts. Missing downloads are flagged separately from corrupt checkpoints.


In [10]:
patch_torch_load_for_lightning()
import torch

GOOD_MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('Drive folder:', DRIVE_GOOD_MODELS_FOLDER_URL)
print('Fallback folder:', FALLBACK_GOOD_MODELS_FOLDER_URL)
print('Expected Drive checkpoints:')
for name, meta in GOOD_MODEL_MANIFEST.items():
    fallback = f" fallback_id={meta['fallback_drive_id']}" if meta.get('fallback_drive_id') else ''
    print(f" - {name} [{meta['family']}] id={meta['drive_id']}{fallback}")

demo_candidates = [GOOD_MODEL_DIR / name for name in GOOD_MODEL_MANIFEST]
loaded_demo = False
errors = []
for demo_ckpt in demo_candidates:
    if not demo_ckpt.exists():
        meta = GOOD_MODEL_MANIFEST[demo_ckpt.name]
        source_hint = meta.get('fallback_drive_id') or meta['drive_id']
        errors.append(f'{demo_ckpt.relative_to(ROOT)} missing; download from Drive id {source_hint}')
        continue
    try:
        ckpt = torch.load(demo_ckpt, map_location='cpu')
        state = ckpt.get('state_dict', ckpt) if isinstance(ckpt, dict) else ckpt
        num_tensors = sum(1 for v in state.values() if hasattr(v, 'shape'))
        num_params = sum(int(v.numel()) for v in state.values() if hasattr(v, 'numel'))
        print('Loaded:', demo_ckpt.relative_to(ROOT))
        print('Tensor entries:', num_tensors)
        print('Parameter count:', num_params)
        print('First keys:', list(state.keys())[:12])
        loaded_demo = num_tensors > 0 and num_params > 0
        if not loaded_demo:
            flag(f'Demo checkpoint {demo_ckpt.name} loaded but did not expose a sane nonempty state dict.')
        break
    except Exception as exc:
        errors.append(f'{demo_ckpt.relative_to(ROOT)}: {type(exc).__name__}: {exc}')

if not loaded_demo:
    flag('No Drive good_models demo checkpoint loaded cleanly: ' + '; '.join(errors))


Drive folder: https://drive.google.com/drive/folders/1nDfQpDWIS8tpA5u2kQHeE1SKDpIZ0t6z
Fallback folder: https://drive.google.com/drive/folders/1sXBohzkTeIUjdRMywBmIzIaVFAwKreno
Expected Drive checkpoints:
 - robust_goat.ckpt [ASVspoof2019] id=1amWa3pEnFdf3d8Z5nU2cqTiSbApjmhob
 - mini_goat-best-epoch=02-val-eer=0.0933.ckpt [ASVspoof2019/mini_goat] id=1p4QQNKtIT24uMyuM3tYzdVvl-uiGKns0
 - mlaad_goat-best-epoch=05-val-eer=0.2795.ckpt [MLAAD regular GOAT] id=19-S-cXCPRDmmW2dWzpolLxt35ub0KCIc
 - mlaad_robust_goat.ckpt [MLAAD robust GOAT] id=1LkLTKsnwLqQia0L7eK4vDRv8pN9IE7Bc fallback_id=1MihkPDsQRvHynU6dLP9UjuyNc-76DJrB
 - mlaad_robust_goat_seed42-best-epoch=05-val-eer=0.3030.ckpt [MLAAD robust GOAT] id=1KMIf1o37fzqDSXJuWJcVQjDDpbteiqgv
 - mlaad_robust_goat_seed1024-best-epoch=03-val-eer=0.2976.ckpt [MLAAD robust GOAT] id=16a6JPg3pZVOoBtynbZPC_iiYn80Zygaq


Loaded: models/good_models/robust_goat.ckpt
Tensor entries: 783
Parameter count: 292019379
First keys: ['model.phoneme_model.model.model.wavlm.masked_spec_embed', 'model.phoneme_model.model.model.wavlm.feature_extractor.conv_layers.0.conv.weight', 'model.phoneme_model.model.model.wavlm.feature_extractor.conv_layers.0.layer_norm.weight', 'model.phoneme_model.model.model.wavlm.feature_extractor.conv_layers.0.layer_norm.bias', 'model.phoneme_model.model.model.wavlm.feature_extractor.conv_layers.1.conv.weight', 'model.phoneme_model.model.model.wavlm.feature_extractor.conv_layers.2.conv.weight', 'model.phoneme_model.model.model.wavlm.feature_extractor.conv_layers.3.conv.weight', 'model.phoneme_model.model.model.wavlm.feature_extractor.conv_layers.4.conv.weight', 'model.phoneme_model.model.model.wavlm.feature_extractor.conv_layers.5.conv.weight', 'model.phoneme_model.model.model.wavlm.feature_extractor.conv_layers.6.conv.weight', 'model.phoneme_model.model.model.wavlm.feature_projection.laye

## Checkpoint QA: All Drive Good Models Submission Checkpoints

**Runs/tests:** CPU load check over the expected `models/good_models/*.ckpt` files from the provided Drive folder.

**Paper/submission claim supported:** every checkpoint planned for submission should load and expose sane tensor counts. Missing files and corrupt files are reported explicitly; ambiguous `e9`, `last`, and smoothing models are listed only if present and are not assumed relevant.


In [11]:
patch_torch_load_for_lightning()
import hashlib
import pandas as pd
import torch

rows = []
for name, meta in GOOD_MODEL_MANIFEST.items():
    path = GOOD_MODEL_DIR / name
    row = {
        'checkpoint': str(path.relative_to(ROOT)),
        'family': meta['family'],
        'drive_id': meta['drive_id'],
        'fallback_drive_id': meta.get('fallback_drive_id', ''),
        'expected_from_drive': True,
        'bytes': path.stat().st_size if path.exists() else 0,
        'ok': False,
        'status': 'missing',
        'error': '',
    }
    if path.exists():
        try:
            obj = torch.load(path, map_location='cpu')
            sd = obj.get('state_dict', obj) if isinstance(obj, dict) else obj
            row['state_tensors'] = sum(1 for v in sd.values() if hasattr(v, 'shape'))
            row['param_count'] = sum(int(v.numel()) for v in sd.values() if hasattr(v, 'numel'))
            row['sha256_12'] = hashlib.sha256(path.read_bytes()).hexdigest()[:12]
            row['ok'] = row['state_tensors'] > 0 and row['param_count'] > 0
            row['status'] = 'loaded' if row['ok'] else 'empty_state_dict'
            del obj, sd
        except Exception as exc:
            row['status'] = 'load_error'
            row['error'] = f'{type(exc).__name__}: {exc}'
            try:
                with path.open('rb') as fh:
                    row['first_8_bytes'] = fh.read(8).hex()
            except Exception:
                pass
    rows.append(row)

ambiguous_rows = []
for search_dir in [GOOD_MODEL_DIR, MODEL_DIR, FRESH_MODEL_DIR, LEGACY_GOOD_MODEL_DIR]:
    if not search_dir.exists():
        continue
    for path in sorted(search_dir.glob('*.ckpt')):
        lower = path.name.lower()
        if any(pattern in lower for pattern in AMBIGUOUS_MODEL_PATTERNS) and path.name not in GOOD_MODEL_MANIFEST:
            ambiguous_rows.append({'checkpoint': str(path.relative_to(ROOT)), 'bytes': path.stat().st_size, 'note': 'ambiguous: inspect before including'})

ckpt_df = pd.DataFrame(rows)
display(ckpt_df)
if ambiguous_rows:
    print('Ambiguous local checkpoints found; not required unless confirmed:')
    display(pd.DataFrame(ambiguous_rows))
else:
    print('No e9/last/smoothing checkpoint files found in the Drive manifest or local checkpoint dirs.')

failed = ckpt_df[~ckpt_df['ok']]
if len(failed):
    flag('Some expected Drive good_models checkpoints are missing or failed to load: ' + failed[['checkpoint', 'status', 'error', 'drive_id', 'fallback_drive_id']].to_string(index=False))
else:
    print('All expected Drive good_models checkpoints loaded and exposed nonempty state dicts.')


,checkpoint,family,drive_id,fallback_drive_id,expected_from_drive,bytes,ok,status,error,state_tensors,param_count,sha256_12
0,models/good_models/robust_goat.ckpt,ASVspoof2019,1amWa3pEnFdf3d8Z5nU2cqTiSbApjmhob,,True,530958106,True,loaded,,783,292019379,184d1c545d56
1,models/good_models/mini_goat-best-epoch=02-val...,ASVspoof2019/mini_goat,1p4QQNKtIT24uMyuM3tYzdVvl-uiGKns0,,True,530967736,True,loaded,,783,292019379,77b9bb9e2d01
2,models/good_models/mlaad_goat-best-epoch=05-va...,MLAAD regular GOAT,19-S-cXCPRDmmW2dWzpolLxt35ub0KCIc,,True,530967736,True,loaded,,783,292019379,38c2ef1033c0
3,models/good_models/mlaad_robust_goat.ckpt,MLAAD robust GOAT,1LkLTKsnwLqQia0L7eK4vDRv8pN9IE7Bc,1MihkPDsQRvHynU6dLP9UjuyNc-76DJrB,True,530967736,True,loaded,,783,292019379,f8869a12d59c
4,models/good_models/mlaad_robust_goat_seed42-be...,MLAAD robust GOAT,1KMIf1o37fzqDSXJuWJcVQjDDpbteiqgv,,True,530967800,True,loaded,,783,292019379,fb3adcd127a3
5,models/good_models/mlaad_robust_goat_seed1024-...,MLAAD robust GOAT,16a6JPg3pZVOoBtynbZPC_iiYn80Zygaq,,True,530967800,True,loaded,,783,292019379,50788b458564


No e9/last/smoothing checkpoint files found in the Drive manifest or local checkpoint dirs.
All expected Drive good_models checkpoints loaded and exposed nonempty state dicts.


## Final Validation Summary

This cell fails only on paper-metric mismatches. Validation flags are printed for unavailable reruns, missing external assets, or checkpoint failures so they can be reported without obscuring the successful metric comparisons.

In [12]:
import pandas as pd
summary = pd.DataFrame(metric_results)
display(summary)
if len(summary) and not bool(summary['ok'].all()):
    raise AssertionError('One or more paper metric checks mismatched. Inspect the table above.')
print('All collected paper metric checks matched their targets within tolerance.')
if validation_flags:
    print('Validation flags:')
    for item in validation_flags:
        print(' -', item)
else:
    print('No validation flags recorded.')


,metric,actual,expected,tol,ok
0,J1 mean-axis fusion dEER,-0.109408,-0.109400,0.0030,True
1,MLAAD WavLM-GAT fusion dEER,-0.109419,-0.109400,0.0030,True
2,mini_goat detector-alone EER,0.143750,0.143750,0.0020,True
3,mini_goat fused EER,0.113750,0.113750,0.0020,True
4,mini_goat dEER,-0.030000,-0.030000,0.0030,True
5,J3 MLAAD LDA n=250 dEER,-0.136055,-0.136055,0.0100,True
6,AASIST zero-shot MLAAD baseline EER,0.375959,0.375959,0.0005,True
7,AASIST zero-shot MLAAD fused EER,0.116490,0.116490,0.0005,True
8,AASIST fine-tuned test EER,0.200451,0.200451,0.0005,True
9,AASIST fine-tuned sd_along rho,0.349498,0.349498,0.0005,True


All collected paper metric checks matched their targets within tolerance.
Validation flags:
 - Axis-production scripts require raw MLAAD wave cache outputs/px_wave_cache/i2_full_test_waves.npz; validating committed artifacts instead.
 - I7 axis-fusion rerun requires raw MLAAD wave cache; validating committed artifact instead.
 - mini_goat rerun needs Hugging Face ASVspoof cache plus loadable GOAT checkpoints; validating committed artifact instead.
 - J3 rerun needs Hugging Face ASVspoof cache and MLAAD wave cache; validating committed artifact instead.
 - AASIST rerun skipped because official external assets are missing: baselines/aasist/models/AASIST.py, baselines/aasist/models/weights/AASIST.pth, baselines/aasist/config/AASIST.conf
 - J4 rerun skipped because /tmp/keys/LA/CM/trial_metadata.txt is missing; validating committed artifact instead.
 - Audit rerun requires raw MLAAD wave cache; validating committed audit outputs instead.


<!-- NEW_DEMO_TRAINING_RECIPES_V1 -->

## Optional Training Recipes

These cells document the repo-native training entry points for regenerating the submitted checkpoints. They are disabled by default so the demo notebook can still be executed end-to-end without launching long GPU jobs. Set `RUN_TRAINING = True` to run them. Set `TRAINING_SMOKE_TEST = True` to run one-batch sanity jobs where the underlying script supports `--fast-dev-run`.

In [13]:
from pathlib import Path
import shlex
import subprocess
import sys

RUN_TRAINING = False
TRAINING_SMOKE_TEST = True

ROOT = Path.cwd().resolve()
assert (ROOT / 'experiments').exists(), f'Run from repo root, got {ROOT}'


def run_training_recipe(name, full_cmd, smoke_cmd=None, expected_outputs=()):
    """Print or run one training recipe.

    This guard is intentional: full model training is expensive and should not run
    during ordinary demo validation. Flip RUN_TRAINING above when intentionally
    regenerating checkpoints.
    """
    cmd = smoke_cmd if TRAINING_SMOKE_TEST and smoke_cmd is not None else full_cmd
    print(f'\n=== {name} ===')
    print('Command:')
    print(' '.join(shlex.quote(str(x)) for x in cmd))
    if not RUN_TRAINING:
        print('Skipped because RUN_TRAINING=False')
    else:
        subprocess.run([str(x) for x in cmd], cwd=ROOT, check=True)
    for output in expected_outputs:
        p = ROOT / output
        print(f'{output}:', 'present' if p.exists() else 'missing')


### Training: mini_goat

Regenerates the deliberately weak mini_goat checkpoint used in the headroom/fusion experiment. The first recipe prepares the tiny ASVspoof-derived train/validation tensors from the Hugging Face cache; the second trains `models/mini_goat.ckpt`.

In [14]:
run_training_recipe(
    'Prepare mini_goat data',
    full_cmd=[sys.executable, 'experiments/results/e_mini_goat_fusion/prepare_mini_goat_data.py'],
    smoke_cmd=None,
    expected_outputs=[
        'experiments/data/mini_goat_processed/splits/train.json',
        'experiments/data/mini_goat_processed/splits/val.json',
    ],
)

run_training_recipe(
    'Train mini_goat',
    full_cmd=[
        sys.executable,
        'experiments/results/e_mini_goat_fusion/train_mini_goat.py',
        '--checkpoint-path', 'models/mini_goat.ckpt',
        '--ckpt-dir', 'experiments/checkpoints',
        '--log-dir', 'experiments/results/e_mini_goat_fusion/training_logs',
    ],
    smoke_cmd=[
        sys.executable,
        'experiments/results/e_mini_goat_fusion/train_mini_goat.py',
        '--fast-dev-run',
        '--epochs', '1',
        '--checkpoint-path', 'models/mini_goat.ckpt',
        '--ckpt-dir', 'experiments/checkpoints',
        '--log-dir', 'experiments/results/e_mini_goat_fusion/training_logs_smoke',
    ],
    expected_outputs=['models/mini_goat.ckpt'],
)



=== Prepare mini_goat data ===
Command:
/opt/conda/bin/python experiments/results/e_mini_goat_fusion/prepare_mini_goat_data.py
Skipped because RUN_TRAINING=False
experiments/data/mini_goat_processed/splits/train.json: missing
experiments/data/mini_goat_processed/splits/val.json: missing

=== Train mini_goat ===
Command:
/opt/conda/bin/python experiments/results/e_mini_goat_fusion/train_mini_goat.py --fast-dev-run --epochs 1 --checkpoint-path models/mini_goat.ckpt --ckpt-dir experiments/checkpoints --log-dir experiments/results/e_mini_goat_fusion/training_logs_smoke
Skipped because RUN_TRAINING=False
models/mini_goat.ckpt: missing


### Training: full ASVspoof robust_goat seeds

Regenerates the full-size ASVspoof-trained robust_goat seed checkpoints. This is the repo's current full-model training path; it writes seed-specific files such as `models/robust_goat_seed7.ckpt`.

In [15]:
for seed in [3, 7]:
    run_training_recipe(
        f'Train full ASVspoof robust_goat seed {seed}',
        full_cmd=[
            sys.executable,
            'experiments/train_new_seed.py',
            '--seed', str(seed),
            '--epochs', '7',
            '--wandb', '0',
        ],
        smoke_cmd=None,
        expected_outputs=[f'models/robust_goat_seed{seed}.ckpt'],
    )



=== Train full ASVspoof robust_goat seed 3 ===
Command:
/opt/conda/bin/python experiments/train_new_seed.py --seed 3 --epochs 7 --wandb 0
Skipped because RUN_TRAINING=False
models/robust_goat_seed3.ckpt: missing

=== Train full ASVspoof robust_goat seed 7 ===
Command:
/opt/conda/bin/python experiments/train_new_seed.py --seed 7 --epochs 7 --wandb 0
Skipped because RUN_TRAINING=False
models/robust_goat_seed7.ckpt: missing


### Training: MLAAD regular and robust_goat

Regenerates the MLAAD-tiny processed dataset and the MLAAD regular/adversarial checkpoints. The adversarial recipe is the one that produces `mlaad_robust_goat`; the regular recipe is included for the paper's regular-vs-robust comparisons.

In [16]:
run_training_recipe(
    'Prepare MLAAD-tiny data',
    full_cmd=[sys.executable, 'experiments/scripts/prepare_mlaad_tiny.py'],
    smoke_cmd=[sys.executable, 'experiments/scripts/prepare_mlaad_tiny.py', '--dry-run'],
    expected_outputs=[
        'experiments/data/mlaad_tiny_processed/splits/train.json',
        'experiments/data/mlaad_tiny_processed/splits/val.json',
        'experiments/data/mlaad_tiny_processed/splits/test.json',
    ],
)

run_training_recipe(
    'Train MLAAD regular goat',
    full_cmd=[
        sys.executable,
        'experiments/scripts/train_mlaad_regular.py',
        '--checkpoint-path', 'experiments/checkpoints/mlaad_goat.ckpt',
        '--log-dir', 'experiments/results/mlaad/training_logs/mlaad_goat',
    ],
    smoke_cmd=[
        sys.executable,
        'experiments/scripts/train_mlaad_regular.py',
        '--fast-dev-run',
        '--epochs', '1',
        '--checkpoint-path', 'experiments/checkpoints/mlaad_goat_smoke.ckpt',
        '--log-dir', 'experiments/results/mlaad/training_logs/mlaad_goat_smoke',
    ],
    expected_outputs=['experiments/checkpoints/mlaad_goat.ckpt'],
)

run_training_recipe(
    'Train MLAAD robust goat',
    full_cmd=[
        sys.executable,
        'experiments/scripts/train_mlaad_adversarial.py',
        '--checkpoint-path', 'models/mlaad_robust_goat.ckpt',
        '--log-dir', 'experiments/results/mlaad/training_logs/mlaad_robust_goat',
    ],
    smoke_cmd=[
        sys.executable,
        'experiments/scripts/train_mlaad_adversarial.py',
        '--fast-dev-run',
        '--epochs', '1',
        '--checkpoint-path', 'experiments/checkpoints/mlaad_robust_goat_smoke.ckpt',
        '--log-dir', 'experiments/results/mlaad/training_logs/mlaad_robust_goat_smoke',
    ],
    expected_outputs=['models/mlaad_robust_goat.ckpt'],
)



=== Prepare MLAAD-tiny data ===
Command:
/opt/conda/bin/python experiments/scripts/prepare_mlaad_tiny.py --dry-run
Skipped because RUN_TRAINING=False
experiments/data/mlaad_tiny_processed/splits/train.json: missing
experiments/data/mlaad_tiny_processed/splits/val.json: missing
experiments/data/mlaad_tiny_processed/splits/test.json: missing

=== Train MLAAD regular goat ===
Command:
/opt/conda/bin/python experiments/scripts/train_mlaad_regular.py --fast-dev-run --epochs 1 --checkpoint-path experiments/checkpoints/mlaad_goat_smoke.ckpt --log-dir experiments/results/mlaad/training_logs/mlaad_goat_smoke
Skipped because RUN_TRAINING=False
experiments/checkpoints/mlaad_goat.ckpt: missing

=== Train MLAAD robust goat ===
Command:
/opt/conda/bin/python experiments/scripts/train_mlaad_adversarial.py --fast-dev-run --epochs 1 --checkpoint-path experiments/checkpoints/mlaad_robust_goat_smoke.ckpt --log-dir experiments/results/mlaad/training_logs/mlaad_robust_goat_smoke
Skipped because RUN_TRAINI